# Here, we will fine-tune a LLAMA model to prevent overfiltering

Here, we import all the libraries that we need to run the code

In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

## Now, we load the dataset to fine-tune model on it

In [ ]:
with open("train.json") as f:
    raw_data = json.load(f)

records = [{"text": d["prompt"], "label": int(d["label"])} for d in raw_data]

dataset = Dataset.from_list(records)
dataset = dataset.class_encode_column("label")
split = dataset.train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
train_ds, eval_ds = split["train"], split["test"]

print(f"Train size: {len(train_ds)}  |  Eval size: {len(eval_ds)}")

## Use this bloct to load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


# keep the raw text length around so group_by_length can bucket similar-length
# examples together (your data ranges from a few words to 8000+ words, so this
# avoids wasting huge amounts of compute padding short sequences up to long ones)
train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)
train_ds = train_ds.map(lambda x: {"length": len(x["input_ids"])})
eval_ds = eval_ds.map(lambda x: {"length": len(x["input_ids"])})
train_ds = train_ds.remove_columns(["text"])
eval_ds = eval_ds.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Load the model

In [ ]:
quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # T4 has no bf16 support
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.pad_token_id

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)  # gradient checkpointing on by default

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()